# 🏨 Hotel Reservation System 9: System-Wide Error Handling

Real agents fail: a database connection drops for a moment, the model API times out, the vector store is down,
or the agent is simply not sure. Some failures fix themselves, and a **retry** is enough. For the others this
notebook gives the core hotel system an **error path**: nodes catch their failures and record them in an `error`
field of the state, a single `error_handler` logs them, and a **decision router** picks a recovery strategy for
each kind of failure. Every failure is injected deterministically, so each recovery path runs on every run.

## Learning Objectives
1. **Errors as state** - Custom exceptions caught inside nodes and stored in an `error` field, not in message text
2. **One logger** - A file logger set up once, and an error handler that logs every failure with the `thread_id` from the config
3. **Recovery strategies** - An alternative model, degraded retrieval with hard-coded rules, and a human handoff (a stub)
4. **Transient failures: `RetryPolicy`** - Re-running a node that fails now and then, with a deterministic failure so the retries are countable
5. **Deterministic failure injection** - `SIMULATE` flags, and a code-computed table proving which recovery ran

## Where this fits
`08_hotel_guest_language_and_sentiment.ipynb` added language and sentiment nodes. This notebook takes the core system of
`03_hotel_compliance_checker.ipynb` from `hotel_common` (without those nodes, to keep the focus on errors) and
makes it survive failures: first transient ones, with a `RetryPolicy` (Part 6), then ones a retry cannot fix, by
switching strategy (Parts 7-8). `10_hotel_web_search_integration.ipynb` adds a web-search agent.

## Prerequisites
- The core system from `03_hotel_compliance_checker.ipynb` (read it first; this notebook imports it from `hotel_common.py`)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`
- The alternative model is `gpt-4o`, so the same key must have access to it

---
## 📦 Step 1: Environment Setup

The setup cell loads `OPENAI_API_KEY` and imports the building blocks and `hotel_common`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

import logging
import uuid
from pathlib import Path
from typing import Literal

from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, RetryPolicy

from datetime import date, timedelta

import hotel_common as hc  # shared hotel building blocks, in this folder (see 01_hotel_environment_and_coordinator.ipynb)
from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: the primary model for all agents
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini", temperature=0)

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🧰 Part 1: Data and Tools

The same database, SQL tools and compliance retriever as notebook 03.

In [ ]:
# ============================================================================
# DATABASE, SQL TOOLS AND THE COMPLIANCE RETRIEVER
# ============================================================================
hc.init_hotel_db("hotel.db")
sql_tools = hc.make_sql_tools("hotel.db")  # 02's tools: run_query only reads, book_room is the only write
retrieve = hc.build_compliance_retriever()
rag_tools = [retrieve]
print(f"✅ {len(sql_tools)} SQL tools, 1 retrieval tool over {len(hc.COMPLIANCE_RULES)} rules")

---
## 🚨 Part 2: Errors as State, and How to Inject Them

- **`HotelState`** adds an `error` field to the messages. A failing node writes a small record there
  (exception type, node, detail) and adds **no** message. Routers check `state["error"]`, so a guest who types
  the word "error" cannot trigger the error path.
- **Three custom exceptions** name the failures this system knows how to recover from.
- **`SIMULATE`** holds one flag per failure. A flag that is on makes that failure happen on every call, so the
  demos below are deterministic. Nothing else in the code is changed to cause a failure.
- **`logs.log`**: the notebook's one file logger. It appends, so the file keeps every run (each run's thread ids
  are unique, so its lines are easy to find). Two details matter: the logger's own level must be set (otherwise it
  inherits `WARNING` and silently drops `INFO` lines), and re-running the cell must not attach a second handler
  (otherwise every line is written twice).

In [ ]:
# ============================================================================
# ERROR STATE, CUSTOM EXCEPTIONS, FAILURE FLAGS AND THE LOG FILE
# ============================================================================
class HotelState(MessagesState):
    error: dict | None  # {"type": exception class name, "node": where it happened, "detail": message}


class LLMCallException(Exception):
    """The coordinator's model could not be called (outage, timeout, rate limit)."""


class RetrievalException(Exception):
    """The compliance vector store could not be searched."""


class NotSureException(Exception):
    """The agent's confidence is too low for it to answer on its own."""


def error_record(exc: Exception, node: str) -> dict:
    return {"type": type(exc).__name__, "node": node, "detail": str(exc)}


# Failure injection: a flag that is True makes that failure happen on every call.
SIMULATE = {"coordinator_llm": False, "retrieval": False, "low_confidence": False}
CONFIDENCE_THRESHOLD = hc.CONFIDENCE_THRESHOLD  # 50, shared with 06: hotel_common.py says why

logger = logging.getLogger("hotel_mas")
logger.setLevel(logging.INFO)              # the logger's own level: without it, INFO lines are dropped
logger.propagate = False                   # write to the file only, not to the notebook output
for old_handler in list(logger.handlers):  # re-running the cell must not add a second handler
    logger.removeHandler(old_handler)
    old_handler.close()
handler = logging.FileHandler("logs.log", mode="a", encoding="utf-8")  # append: the file keeps every run
handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(handler)

print(f"✅ Failure flags: {SIMULATE}")
print("✅ Errors are logged to", Path("logs.log").resolve())

---
## 🤖 Part 3: Agents That Catch Their Own Failures

The reservation agent and `generate` come from `hc.make_core_nodes()` (03's agent, 04's structured verdict).
`generate` reads its rules from the `retrieve` tool's results or, in degraded mode, from the hard-coded rules
message (see Part 4). The two agents that can fail here are written out, with a `try/except` around the part
that can fail:
- **`conv_assistant`** checks its confidence first (`NotSureException` below the threshold). The confidence is a
  stand-in value here, set by the `low_confidence` flag; notebook 05 showed how to get it from the model's own
  self-assessment. It then calls the model through `call_coordinator_llm()`, which turns *any* failure, real or
  simulated, into an `LLMCallException`.
- **`compliance_checker`** raises `RetrievalException` when the vector store is unavailable. The flag injects the
  failure at the node; a production version would also wrap the tool call itself.

In [ ]:
# ============================================================================
# AGENT NODES THAT RECORD THEIR FAILURES
# ============================================================================
AGENTS = ["reservation_assistant", "compliance_checker"]
conversation_prompt = hc.conversation_prompt(AGENTS, extra_guidelines=hc.COMPLIANCE_FIRST)  # 03's guideline
sql_assistant, _, generate = hc.make_core_nodes(llm, sql_tools, rag_tools)  # 03's reservation agent, 04's verdict
llm_with_rag_tools = llm.bind_tools(rag_tools)


def call_coordinator_llm(messages):
    """Call the primary model; any failure, real or simulated, becomes an LLMCallException."""
    if SIMULATE["coordinator_llm"]:
        raise LLMCallException("Simulated outage of the primary model")
    try:
        return llm.invoke(messages)
    except Exception as e:
        raise LLMCallException(f"Primary model call failed: {e}") from e


def conv_assistant(state: HotelState) -> dict:
    try:
        confidence = 40 if SIMULATE["low_confidence"] else 90  # stand-in for a self-assessed confidence
        if confidence < CONFIDENCE_THRESHOLD:
            raise NotSureException(f"Confidence {confidence} is below the threshold of {CONFIDENCE_THRESHOLD}")
        reply = call_coordinator_llm([conversation_prompt] + state["messages"])
    except (LLMCallException, NotSureException) as e:
        return {"error": error_record(e, "conv_assistant")}
    return {"messages": [reply], "error": None}


def rag_assistant(state: HotelState) -> dict:
    try:
        if SIMULATE["retrieval"]:
            raise RetrievalException("Simulated outage: the compliance vector store is unreachable")
        reply = llm_with_rag_tools.invoke([hc.COMPLIANCE_PROMPT] + state["messages"])
    except RetrievalException as e:
        return {"error": error_record(e, "compliance_checker")}
    return {"messages": [reply], "error": None}


print("✅ Agent nodes defined")

---
## 🛟 Part 4: The Error Handler and Three Recovery Strategies

| Failure | Recovery node | What it does |
|---|---|---|
| `LLMCallException` | `alternative_model` | answers as the coordinator with a second model (`gpt-4o`) |
| `RetrievalException` | `degraded_retrieval` | skips the vector store and hands the hard-coded rules to `generate` with `Command(goto="generate")` |
| anything else (`NotSureException`, a failing fallback) | `human_intervention` | a **stub**: it only writes a hand-off message (a real system would open a ticket, or pause with `interrupt()` as in 04) |

`error_handler` logs the failure together with the `thread_id`, which it reads from the `RunnableConfig`
LangGraph passes to every node that asks for it.
`decision_router` is a pure function of `state["error"]`. If the alternative model fails too, its error type is
not `LLMCallException`, so the router sends it to a human instead of looping.

In [ ]:
# ============================================================================
# ERROR HANDLER, DECISION ROUTER AND RECOVERY NODES
# ============================================================================
alternative_llm = get_openai_llm(model="gpt-4o", temperature=0)


def error_handler(state: HotelState, config: RunnableConfig) -> dict:
    err = state["error"]
    thread_id = config["configurable"]["thread_id"]
    logger.error(f"{err['type']} in {err['node']}: {err['detail']} | thread_id={thread_id}")
    print(f"   ⚠️ error_handler: {err['type']} in {err['node']} (logged for thread {thread_id})")
    return {"error": err}  # unchanged, but written, so the step shows up in stream_mode="updates"


RECOVERY = {"LLMCallException": "alternative_model", "RetrievalException": "degraded_retrieval"}


def decision_router(state: HotelState) -> Literal["alternative_model", "degraded_retrieval", "human_intervention"]:
    return RECOVERY.get(state["error"]["type"], "human_intervention")


def alternative_model(state: HotelState) -> dict:
    print(f"   🔁 alternative_model: answering with {alternative_llm.model_name}")
    try:
        reply = alternative_llm.invoke([conversation_prompt] + state["messages"])
    except Exception as e:  # its type is not LLMCallException, so the router escalates to a human
        return {"error": error_record(e, "alternative_model")}
    return {"messages": [reply], "error": None}


def degraded_retrieval(state: HotelState) -> Command[Literal["generate"]]:
    print("   📉 degraded_retrieval: using the hard-coded rules instead of the vector store")
    rules = "\n".join(f"Rule {i}: {rule}" for i, rule in enumerate(hc.COMPLIANCE_RULES, start=1))
    fallback = AIMessage(content=f"(Vector store unavailable, all rules listed)\n{rules}", name="fallback_rules")
    return Command(update={"messages": [fallback], "error": None}, goto="generate")


def human_intervention(state: HotelState) -> dict:
    """A STUB: it only writes a hand-off message. A real system would open a ticket, or pause the graph with
    interrupt() until a person answers, as 04_hotel_edge_cases_and_errors.ipynb shows."""
    print("   🙋 human_intervention (a stub): writing the staff hand-off message")
    handoff = AIMessage(
        content="I'm sorry, I can't handle this reliably right now. A member of our staff will take over shortly.",
        name="human_handoff",
    )
    return {"messages": [handoff], "error": None}


print("✅ Recovery nodes defined")

---
## 🔀 Part 5: Routing on the `error` Field, and the Graph

Both routers check `state["error"]` **before** anything else. After that, `choose_next_node` is
`hc.route_by_mention()`, 03's router with its loop guard: once the latest request has a verdict,
`compliance_checker` is no longer a candidate, so a reply that says "the compliance checker approved it,
forwarding to reservation_assistant" still reaches the reservation agent.

`build_graph()` takes the reservation node and a retry policy as arguments, so Part 6 can swap in a node that fails.

In [ ]:
# ============================================================================
# ROUTERS AND THE GRAPH BUILDER
# ============================================================================
def choose_next_node(state: HotelState) -> Literal["error_handler", "reservation_assistant", "compliance_checker", "__end__"]:
    if state.get("error"):
        return "error_handler"
    return hc.route_by_mention(state["messages"], AGENTS) or END  # 03's router, with its loop guard


def after_compliance_checker(state: HotelState) -> Literal["error_handler", "rag_tools", "conv_assistant"]:
    if state.get("error"):
        return "error_handler"
    return "rag_tools" if getattr(state["messages"][-1], "tool_calls", None) else "conv_assistant"


COORDINATOR_TARGETS = ["error_handler", "reservation_assistant", "compliance_checker", END]


def build_graph(reservation_node=sql_assistant, retry_policy=None):
    builder = StateGraph(HotelState)
    builder.add_node("conv_assistant", conv_assistant)
    builder.add_node("reservation_assistant", reservation_node, retry_policy=retry_policy)
    builder.add_node("compliance_checker", rag_assistant)
    builder.add_node("generate", generate)
    builder.add_node("sql_tools", ToolNode(sql_tools))
    builder.add_node("rag_tools", ToolNode(rag_tools))
    builder.add_node("error_handler", error_handler)
    builder.add_node("alternative_model", alternative_model)
    builder.add_node("degraded_retrieval", degraded_retrieval)  # routes itself with Command(goto="generate")
    builder.add_node("human_intervention", human_intervention)

    builder.add_edge(START, "conv_assistant")
    builder.add_conditional_edges("conv_assistant", choose_next_node, COORDINATOR_TARGETS)
    builder.add_conditional_edges("alternative_model", choose_next_node, COORDINATOR_TARGETS)
    builder.add_conditional_edges("reservation_assistant", tools_condition, {"tools": "sql_tools", END: END})
    builder.add_conditional_edges("compliance_checker", after_compliance_checker, ["error_handler", "rag_tools", "conv_assistant"])
    builder.add_edge("sql_tools", "reservation_assistant")
    builder.add_edge("rag_tools", "generate")
    builder.add_edge("generate", "conv_assistant")
    builder.add_conditional_edges("error_handler", decision_router, ["alternative_model", "degraded_retrieval", "human_intervention"])
    builder.add_edge("human_intervention", END)
    return builder.compile(checkpointer=InMemorySaver())


graph = build_graph()
print("✅ Graph compiled. Edges:")
hc.print_edges(graph)

---
## 🔁 Part 6: Transient Failures: Retry

Some failures fix themselves: a database connection drops for a moment, an API times out once. The cheapest
recovery is to run the node again. `RetryPolicy` re-runs a node when it raises, with a growing wait between
attempts. A small wrapper makes the failure **deterministic**: the reservation node raises a `ConnectionError` on
its first `fail_first_n` calls and works after that, records every attempt and logs each failure.

`RetryPolicy` retries only exceptions its `retry_on` accepts. The default already covers `ConnectionError`; it is
spelled out here to make the choice visible. This failure is *raised*, not recorded in `state["error"]`: the
error path of Parts 2-5 is for failures a retry cannot fix.

In [ ]:
# ============================================================================
# A NODE THAT FAILS ITS FIRST CALLS
# ============================================================================
class FlakyReservationNode:
    """Wraps a node: raises ConnectionError on the first `fail_first_n` calls, then delegates. Records each call."""

    def __init__(self, node, fail_first_n: int):
        self.node, self.fail_first_n, self.calls = node, fail_first_n, []

    def __call__(self, state: dict) -> dict:
        n = len(self.calls) + 1
        if n <= self.fail_first_n:
            self.calls.append("fail")
            logger.error("reservation_assistant call %d failed: simulated connection error", n)
            raise ConnectionError(f"simulated: database unreachable (call {n})")
        self.calls.append("ok")
        return self.node(state)


retry = RetryPolicy(max_attempts=3, initial_interval=0.2, jitter=False, retry_on=ConnectionError)
print(retry)

### Without and with a retry policy
The same request and the same two failures, once on a graph without a policy and once with
`RetryPolicy(max_attempts=3)`.

In [ ]:
# ============================================================================
# TWO FAILURES, WITHOUT AND WITH RETRIES
# ============================================================================
stay = date.today() + timedelta(days=75)
request = f"I'm Sam Ortiz and I'd like to book a Suite from {stay} to {stay + timedelta(days=2)}."

no_retry = FlakyReservationNode(sql_assistant, fail_first_n=2)
try:
    hc.run_turn(build_graph(no_retry), request, thread_id="retry-off")
    crashed = False
except ConnectionError as error:
    crashed = True
    print(f"💥 The run stopped: {error}")
print("=" * 80)

with_retry = FlakyReservationNode(sql_assistant, fail_first_n=2)
hc.run_turn(build_graph(with_retry, retry_policy=retry), request, thread_id="retry-on")

print("\n--- Checks ---")
print(f"{'✅' if crashed and no_retry.calls == ['fail'] else '❌'} without a policy: calls={no_retry.calls}, run crashed")
print(f"{'✅' if with_retry.calls[:3] == ['fail', 'fail', 'ok'] else '❌'} with RetryPolicy(max_attempts=3): "
      f"calls={with_retry.calls}")

**Reading the output:** without a policy the first `ConnectionError` ends the whole run (`calls=['fail']`). With
the policy the node fails twice, waits 0.2 s and then 0.4 s, and succeeds on the third attempt: the guest never
sees the failures. Extra `'ok'` entries after the third are the agent's normal returns from its SQL tools. If
`calls=[]`, the coordinator never forwarded the request to the reservation agent, so the node never ran.

With `fail_first_n=3` the third attempt fails too and the exception reaches the caller: a retry policy smooths
over brief outages; it does not replace error handling. The rest of this notebook handles the failures a retry
cannot fix.

---
## ▶️ Part 7: Failures a Retry Cannot Fix, One Turn Each

The same guest question runs four times, each on a fresh thread: once with no failure, then once with each flag
switched on. `run_scenario()` turns the flag on only for that turn (and off again even if the turn crashes) and
records which nodes ran from `stream_mode="updates"`.

Note that the `coordinator_llm` outage lasts the whole turn: every coordinator call in it falls back, so
`alternative_model` can appear more than once in the path.

In [ ]:
# ============================================================================
# RUN EACH SCENARIO AND RECORD THE PATH
# ============================================================================
RECOVERY_NODES = ["alternative_model", "degraded_retrieval", "human_intervention"]
QUESTION = "Do you have any rooms with a jacuzzi?"
scenarios = [  # (name, flag to switch on, recovery node expected to run)
    ("baseline", None, None),
    ("coordinator_llm", "coordinator_llm", "alternative_model"),
    ("retrieval", "retrieval", "degraded_retrieval"),
    ("low_confidence", "low_confidence", "human_intervention"),
]


def run_scenario(name: str, flag: str | None) -> tuple[str, list[str]]:
    thread_id = f"errors-{name}-{uuid.uuid4().hex[:6]}"
    config = {"configurable": {"thread_id": thread_id}, "recursion_limit": 30}
    for key in SIMULATE:
        SIMULATE[key] = key == flag
    nodes = []
    try:
        for chunk in graph.stream({"messages": [HumanMessage(content=QUESTION)]}, config, stream_mode="updates"):
            nodes.extend(chunk.keys())
    finally:
        for key in SIMULATE:
            SIMULATE[key] = False
    final = graph.get_state(config).values["messages"][-1]
    print(f"path: {' → '.join(nodes)}\n🤖 {final.content}\n")
    return thread_id, nodes


results = []
for name, flag, expected in scenarios:
    print(f"━━━ Scenario: {name} (injected: {flag or 'nothing'}) ━━━")
    thread_id, nodes = run_scenario(name, flag)
    results.append((name, flag, expected, thread_id, nodes))

### Which recovery really ran?
The table is computed from the recorded node paths, not from the replies.

In [ ]:
# ============================================================================
# INJECTED FAILURE -> RECOVERY NODE THAT RAN
# ============================================================================
print(f"{'Injected failure':<18} {'Expected recovery':<20} {'Recovery nodes that ran':<42} OK")
for name, flag, expected, thread_id, nodes in results:
    ran = list(dict.fromkeys(n for n in nodes if n in RECOVERY_NODES))  # unique, in order
    if expected:
        ok = expected in ran
    else:
        ok = not any(n in nodes for n in ["error_handler"] + RECOVERY_NODES)  # baseline: no error path at all
    print(f"{flag or 'none':<18} {expected or 'none':<20} {', '.join(ran) or '-':<42} {'✅' if ok else '❌'}")

retrieval_nodes = next(r[4] for r in results if r[0] == "retrieval")
if "compliance_checker" not in retrieval_nodes:
    print("\nNote: in the retrieval scenario the coordinator never forwarded to compliance_checker, so the "
          "retrieval failure could not happen.")

**Reading the output:** each ✅ line pairs an injected failure with the recovery node that actually ran. The
baseline row has no recovery and no `error_handler` step. Compare the replies above: the `retrieval` scenario
still produced a compliance verdict (from the hard-coded rules), the `coordinator_llm` scenario still answered
the guest (from `gpt-4o`), and `low_confidence` ended with the staff handoff message. A ❌ in the retrieval row
together with the note means the coordinator skipped the compliance check, which the prompt asks for but does
not force.

---
## 📜 Part 8: The Log File

Every failure went through `error_handler`, so each failure scenario should have a line in `logs.log` carrying
its own `thread_id`: the value the node read from the config. (Part 6's simulated connection errors are in the
file too, as `ERROR` lines.)

In [ ]:
# ============================================================================
# READ LOGS.LOG AND CHECK ONE ENTRY PER FAILURE
# ============================================================================
lines = Path("logs.log").read_text().splitlines()
print("Last lines of logs.log:")
for line in lines[-5:]:
    print("  " + line)

print()
for name, flag, expected, thread_id, nodes in results:
    if flag:
        logged = any(f"thread_id={thread_id}" in line for line in lines)
        print(f"{'✅' if logged else '❌'} {name}: logged with thread_id={thread_id}")

**Reading the output:** the log keeps growing across runs (it is a file, not state), so look at the thread ids:
they are unique per run, and every failure scenario's id appears in it. That is the handle an on-call engineer
needs to find the conversation in the checkpointer.

To experiment yourself, switch a flag on (`SIMULATE["retrieval"] = True`) and chat with
`hc.chat_with_agent(graph, "my-thread")` in a new cell.

---
## 📝 Summary

- Nodes **catch** their failures and write an `error` record to state; routers check that field, never the
  message text.
- One file **logger**, set up once (its own level, no duplicate handler, append mode); `error_handler` logs every
  failure with the `thread_id` from the `RunnableConfig`.
- A pure **`decision_router`** maps the error type to a recovery: an alternative model, degraded retrieval
  (`Command(goto=...)` straight to the verdict node) or a human handoff (a stub here), which also catches failing
  fallbacks.
- **`RetryPolicy(max_attempts=...)`** re-runs a node after a transient failure; a deterministic failure made the
  retries countable. It smooths over brief outages and does not replace the error path.
- **`SIMULATE` flags** make each failure deterministic, and the path recorded with `stream_mode="updates"`
  proves which recovery ran.

### Next Steps
- `10_hotel_web_search_integration.ipynb`: a web-search agent that answers questions about the hotel's
  neighbourhood